In [1]:
import napari
import numpy as np
from aicsimageio import AICSImage

import pandas as pd
from skimage import measure


import tifffile as tf
import matplotlib.pyplot as plt
import xarray as xr
from tifffile import tifffile
import tifftools
import os
from matplotlib import pyplot as plt
from os.path import sep
from skimage import io
from PIL import Image
import imageio


from skimage.measure import regionprops_table

import seaborn as sns

import czifile

import math

from pathlib import Path
import re

from scipy.fft import fft, fftfreq
from scipy.optimize import curve_fit


/Users/fisherguest/miniconda3/envs/sansachen_czi/lib/python3.11/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [19]:
glomeruli = ["L9","L8","L7","L6","L5","L4","L3","L2","L1",
             "R1","R2","R3","R4","R5","R6","R7","R8","R9"]

In [20]:
files = {
    "fly1": "/Users/fisherguest/Documents/Projects/connectome_analysis_fisherlab/single cell fly csv/fly1.csv",
    "fly2": "/Users/fisherguest/Documents/Projects/connectome_analysis_fisherlab/single cell fly csv/fly2.csv",
    #"fly3": "/Users/fisherguest/Documents/Projects/connectome_analysis_fisherlab/single cell fly csv/fly3.csv"
}

In [21]:
original = {
    name: pd.read_csv(path, index_col=0).assign(glomerulus=glomeruli)[
        ["glomerulus","mean_red","mean_green","total_red","total_green"]
    ]
    for name, path in files.items()
}

In [24]:
original["fly1"]

,glomerulus,mean_red,mean_green,total_red,total_green
0,L9,848.954201,73.941009,4.022761e+08,35036873.0
1,L8,955.666263,71.846106,6.839656e+08,51419899.0
2,L7,846.518062,63.483875,5.785502e+08,43387864.0
3,L6,888.429529,59.067325,3.275302e+08,21775878.0
4,L5,849.205255,61.282188,3.313701e+08,23913045.0
5,L4,769.550523,56.653493,2.707810e+08,19934608.0
6,L3,1429.598217,80.145079,2.727652e+09,152915609.0
7,L2,615.791312,45.274323,2.307241e+08,16963338.0
8,L1,723.791525,54.077473,4.096240e+08,30604713.0
9,R1,796.183173,52.765766,1.056030e+09,69986718.0


In [23]:
# another (previous) way to open all cvs one by one
'''
fly1 = pd.read_csv("/Users/fisherguest/Documents/Projects/connectome_analysis_fisherlab/single cell fly csv/fly1.csv", index_col=0).assign(glomerulus=glomeruli)[["glomerulus","mean_red","mean_green","total_red","total_green"]]
fly2 = pd.read_csv("/Users/fisherguest/Documents/Projects/connectome_analysis_fisherlab/single cell fly csv/fly2.csv", index_col=0).assign(glomerulus=glomeruli)[["glomerulus","mean_red","mean_green","total_red","total_green"]]
'''


'\nfly1 = pd.read_csv("/Users/fisherguest/Documents/Projects/connectome_analysis_fisherlab/single cell fly csv/fly1.csv", index_col=0).assign(glomerulus=glomeruli)[["glomerulus","mean_red","mean_green","total_red","total_green"]]\nfly2 = pd.read_csv("/Users/fisherguest/Documents/Projects/connectome_analysis_fisherlab/single cell fly csv/fly2.csv", index_col=0).assign(glomerulus=glomeruli)[["glomerulus","mean_red","mean_green","total_red","total_green"]]\n'

In [35]:
fly_subtype = pd.DataFrame({
    'fly': ['fly1','fly2'],
    'subtype': ['L3R6','L4R5']
})

In [36]:
tokens = fly_subtype['subtype'].str.findall(r'[LR]\d+')

# 2. Figure out the maximum number of tokens any row has
max_parts = tokens.map(len).max()

# 3. Create column names dynamically (part1, part2, …)
col_names = [f'part{i+1}' for i in range(max_parts)]

# 4. “Unpack” the lists of tokens into separate columns
fly_subtype[col_names] = pd.DataFrame(tokens.tolist(), index=fly_subtype.index)

fly_subtype

,fly,subtype,part1,part2
0,fly1,L3R6,L3,R6
1,fly2,L4R5,L4,R5


In [26]:
brains_with_3glo = {}
brains_with_2glo = {}

for _, row in fly_subtype.iterrows():
    fly_name = row["fly"]
    subtype  = row["subtype"]

    if len(subtype) == 4:
        brains_with_2glo[fly_name] = original[fly_name]
    else:
        brains_with_3glo[fly_name] = original[fly_name]

# to check
list(brains_with_3glo.keys())
list(brains_with_2glo.keys())

['fly1', 'fly2']

In [62]:
brains_with_2glo_halves = {}

# Iterate over your 2-glom dict
for name, df in brains_with_2glo.items():
    # slice out the first half and second half
    df_first  = df.iloc[:9].reset_index(drop=True)
    df_second = df.iloc[9:].reset_index(drop=True)
    
    # build new keys
    name_R = f"{name}_R"   # e.g. "delta7_911565419_L3R6_R_upstream_R"
    name_L = f"{name}_L"   # e.g. "delta7_911565419_L3R6_R_upstream_L"
    
    # store them
    brains_with_2glo_halves[name_R] = df_second
    brains_with_2glo_halves[name_L] = df_first

In [63]:
brains_with_2glo_halves["fly1_L"]

,glomerulus,mean_red,mean_green,total_red,total_green
0,L9,848.954201,73.941009,4.022761e+08,35036873.0
1,L8,955.666263,71.846106,6.839656e+08,51419899.0
2,L7,846.518062,63.483875,5.785502e+08,43387864.0
3,L6,888.429529,59.067325,3.275302e+08,21775878.0
4,L5,849.205255,61.282188,3.313701e+08,23913045.0
5,L4,769.550523,56.653493,2.707810e+08,19934608.0
6,L3,1429.598217,80.145079,2.727652e+09,152915609.0
7,L2,615.791312,45.274323,2.307241e+08,16963338.0
8,L1,723.791525,54.077473,4.096240e+08,30604713.0


In [38]:
list(brains_with_2glo_halves.keys())

['fly1_R', 'fly1_L', 'fly2_R', 'fly2_L']

In [37]:
# add cvs to brains_with_3glo_halves
# ...

In [ ]:
# fast lookup from fly_subtype
parts_lookup = fly_subtype.set_index("fly")[["part1", "part2"]].to_dict("index")

reordered_2glo_halves = {}

for name, df in brains_with_2glo_halves.items():
    base_name = name.split("_")[0]                   # e.g., "fly1_L" -> "fly1"
    column_to_look = "part1" if "_L" in name else "part2"

    parts = parts_lookup.get(base_name)
    if parts is None or pd.isna(parts.get(column_to_look, None)):
        continue

    center = parts[column_to_look]                   # e.g., "R4" or "L3"

    # find the row index where glomerulus == center
    idxs = df.index[df["glomerulus"] == center]
    if len(idxs) == 0:                              # skip if don't need to reorder
        continue
    current_idx = int(idxs[0])

    # put center on the 5th row (0-based index 4)
    target_idx = 4
    shift = target_idx - current_idx

    # roll row order safely
    order = np.roll(np.arange(len(df)), shift)
    df_reordered = df.iloc[order].reset_index(drop=True)

    reordered_2glo_halves[f"{name}_reordered"] = df_reordered


In [71]:
list(reordered_2glo_halves.keys())

['fly1_R_reordered',
 'fly1_L_reordered',
 'fly2_R_reordered',
 'fly2_L_reordered']

In [73]:
reordered_2glo_halves['fly1_L_reordered']

,glomerulus,mean_red,mean_green,total_red,total_green
0,L7,846.518062,63.483875,5.785502e+08,43387864.0
1,L6,888.429529,59.067325,3.275302e+08,21775878.0
2,L5,849.205255,61.282188,3.313701e+08,23913045.0
3,L4,769.550523,56.653493,2.707810e+08,19934608.0
4,L3,1429.598217,80.145079,2.727652e+09,152915609.0
5,L2,615.791312,45.274323,2.307241e+08,16963338.0
6,L1,723.791525,54.077473,4.096240e+08,30604713.0
7,L9,848.954201,73.941009,4.022761e+08,35036873.0
8,L8,955.666263,71.846106,6.839656e+08,51419899.0


In [76]:
# need to do the same for reordered_3glo_halves
reordered_3glo_halves = {}

# ...

In [77]:
# maybe combine reordered_2glo_halves and reordered_3glo_halves into a reordered_dfs
reordered_halves_all = {**reordered_2glo_halves, **reordered_3glo_halves}
list(reordered_halves_all.keys())

['fly1_R_reordered',
 'fly1_L_reordered',
 'fly2_R_reordered',
 'fly2_L_reordered']

In [ ]:
for name, df in reordered_halves_all.items():
    # 1) pull out the code section between the 2nd and 3rd underscore
    parts = name.split('_')
    code_section = parts[2]  # e.g. "L3R6" (length 4) or "L3L4R6" (length 6)

    # 2) decide which row‐indices to color red
    if len(code_section) == 4:
        # pattern L*R* → only index 4 in red
        red_indices = [4]
    elif len(code_section) == 6:
        # pattern L*L*R* or L*R*R* → indices 3 and 4 in red
        red_indices = [3, 4]
    else:
        # fallback
        red_indices = []

    # 3) prepare data
    regions = df['Glomerulus'].tolist()
    counts  = df['SynapseCount'].astype(float).tolist()
    x       = np.arange(len(regions))

    # 4) build a color list
    colors = ['grey'] * len(regions)
    for idx in red_indices:
        if 0 <= idx < len(colors):
            colors[idx] = 'red'

    # 5) plot
    plt.figure(figsize=(6, 4))
    plt.bar(x, counts, color=colors)
    plt.xticks(x, regions, rotation=45)
    plt.xlabel('')              # remove x-axis label
    plt.ylabel('SynapseCount')
    plt.title(name)
    plt.ylim(0, 90) 
    plt.tight_layout()
    plt.show()